In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "eval" / "retrieval_smoke.py").is_file():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not find the mini-rag project root")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from eval.retrieval_smoke import load_questions_async
from openai import AsyncOpenAI
from rich_struct import print_obj

from mini_rag.ask import ask_async
from mini_rag.config import DEFAULT_EMBEDDING_MODEL, AskConfig
from mini_rag.llm import OpenAIAnswerGenerator, OpenAIRetriever
from mini_rag.models import Answer


In [2]:
async def answer_question(question: str) -> Answer:
    config = AskConfig(log_folder=Path("../logs"), question=question, index_dir=Path("../index") / DEFAULT_EMBEDDING_MODEL)

    async with AsyncOpenAI() as client:
        retriever = OpenAIRetriever(config, client)
        await retriever.load_index(Path("../index/chunks_600_90.jsonl"))

        generator = OpenAIAnswerGenerator(client, config.answer_model)

        answer = await ask_async(config.question, retriever, generator, config.top_k)
        return answer

In [3]:
questions = [q async for q in load_questions_async(Path("../data/questions.jsonl"))]

In [4]:
answer_ok = await answer_question(questions[1].question)
print_obj(answer_ok)

22:58:52.406 INFO     Loading index data from ..\index\text-embedding-3-small
22:58:52.408 INFO     Loading embeddings manifest from ..\index\text-embedding-3-small\embedding_manifest.json
22:58:52.430 INFO     Loading normalized embeddings from ..\index\text-embedding-3-small\embeddings.npy
22:58:52.456 INFO     Reading chunks from ..\index\chunks_600_90.jsonl
22:58:53.207 INFO     Retrieving closest semanticly chunks for user question
22:58:53.209 DEBUG    Searching for text What is the difference between climate mitigation and adaptation?
22:58:53.210 INFO     Calling OpenAI embeddings (model=text-embedding-3-small) with 1 strings [Batch: 1]
22:58:55.128 DEBUG    OpenAI returned 1 embeddings (total_tokens: 10) [Batch: 1]
22:58:55.130 DEBUG    Semantic searching
22:58:55.134 INFO     Building question context
22:58:55.135 INFO     Getting answer for user question from LLM
22:58:55.136 INFO     Quering OpenAI's model 'gpt-5.4-mini'
22:58:58.032 INFO     OpenAI model 'gpt-5.4-mini' responded in 2.89 seconds (input_tokens: 2726, output_tokens: 108, total_tokens: 2834) [Response: resp_0f8e011b5895d586006aff009edc9887d08d406894ec31f457]

Answer(                                                                                                            
    answer='Climate mitigation refers to actions aimed at reducing greenhouse gas emissions or otherwise addressing
the causes of climate change, while adaptation refers to adjusting human or natural systems to actual or expected  
climate and its effects in order to reduce harm or take advantage of opportunities. [1][5] The fragments also note 
that mitigation and adaptation can overlap and interact, but they are distinct policy areas. [1][4]',              
    citations=[                                                                                                    
        Citation(                                                                                                  
            ref=1,                                                                                                 
            chunk_id='WGIII_CH14_p047_c002',                                                                       
            document_id='WGIII_CH14',                                                                              
            page=47,                                                                                               
            page_end=47                                                                                            
        ),                                                                                                         
        Citation(                                                                                                  
            ref=5,                                                                                                 
            chunk_id='WGIII_CH04_p060_c003',                                                                       
            document_id='WGIII_CH04',                                                                              
            page=60,                                                                                               
            page_end=60                                                                                            
        ),                                                                                                         
        Citation(                                                                                                  
            ref=4,                                                                                                 
            chunk_id='WGIII_CH13_p049_c001',                                                                       
            document_id='WGIII_CH13',                                                                              
            page=49,                                                                                               
            page_end=51                                                                                            
        )                                                                                                          
    ],                                                                                                             
    insufficient_evidence=False,                                                                                   
    retrieved_chunk_ids=[                                                                                          
        'WGIII_CH14_p047_c002',                                                                                    
        'WGIII_CH04_p060_c003',                                                                                    
        'WGIII_CH13_p049_c001'                                                                                     
    ]                                                                                                              
)                                                       

In [5]:
answer_fail = await answer_question(questions[28].question)
print_obj(answer_fail)

23:00:59.281 INFO     Loading index data from ..\index\text-embedding-3-small
23:00:59.283 INFO     Loading embeddings manifest from ..\index\text-embedding-3-small\embedding_manifest.json
23:00:59.300 INFO     Loading normalized embeddings from ..\index\text-embedding-3-small\embeddings.npy
23:00:59.321 INFO     Reading chunks from ..\index\chunks_600_90.jsonl
23:01:00.124 INFO     Retrieving closest semanticly chunks for user question
23:01:00.126 DEBUG    Searching for text How much did Apple earn in fiscal year 2025?
23:01:00.127 INFO     Calling OpenAI embeddings (model=text-embedding-3-small) with 1 strings [Batch: 1]
23:01:01.368 DEBUG    OpenAI returned 1 embeddings (total_tokens: 12) [Batch: 1]
23:01:01.369 DEBUG    Semantic searching
23:01:01.373 INFO     Building question context
23:01:01.374 INFO     Getting answer for user question from LLM
23:01:01.375 INFO     Quering OpenAI's model 'gpt-5.4-mini'
23:01:03.641 INFO     OpenAI model 'gpt-5.4-mini' responded in 2.26 seconds (input_tokens: 3145, output_tokens: 38, total_tokens: 3183) [Response: resp_05d85a82026f3350006aff011cf45887d0ab1266cc3ce9ada1]

Answer(                                                                                                            
    answer="I'm sorry but based on provided document fragments I can't answer the question",                       
    citations=[],                                                                                                  
    insufficient_evidence=True,                                                                                    
    retrieved_chunk_ids=[]                                                                                         
)                                                                                                                  